# New hand-annotated complex-call folders (temp)

Quick look at the two folders added in July 2026:

| folder | what it is |
|:-|:-|
| `2026_07_10_new_data` | putative **call sequences** — like the data we've been looking at |
| `2026_07_15__synchronizing_calls_examples` | the **opposite**: different calls that appear to come from **two different animals** |

Each folder is plotted in its own figure. **Every row uses the exact same x-axis**
(same window width, aligned on the first annotated call at `t = 0`) so rows can be
compared directly by eye.


In [ ]:
# === Setup: run once ===
import sys
from pathlib import Path

import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Rectangle
from scipy.io import wavfile

REPO_ROOT = Path("/mnt/home/gginosar/repos/gerbil_vocalization_analysis")
sys.path.insert(0, str(REPO_ROOT / "scripts" / "utils"))

from spectrogram_viz import read_audio, plot_spectrogram, plot_segments_overlay

COMPLEX_ROOT = Path(
    "/mnt/home/neurostatslab/ceph/saneslab_data/gily_data/"
    "Pre_processing/das/models/TRAINING_data_complex"
)
SEQ_DIR  = COMPLEX_ROOT / "2026_07_10_new_data"                       # call sequences (one animal)
SYNC_DIR = COMPLEX_ROOT / "2026_07_15__synchronizing_calls_examples"  # two animals together

# --- Shared plot settings. These are identical for every row in every figure, ---
# --- which is what makes the panels comparable.                              ---
WINDOW_S = 4.0      # seconds per row -- the shared x-axis width
PRE_S    = 0.25     # shown before the first annotated call
POST_S   = 0.25     # shown after the last annotated call
ROW_H    = 2.0      # inches per row
FMIN_HZ  = 1_000
FMAX_HZ  = 60_000
VMIN_DB, VMAX_DB = -40.0, 0.0

# Each file is WRAPPED across as many rows as it needs, so nothing is hidden.
#   "blocks"    -> ONLY the annotated stretches; silence between them is cut out
#                  and the pieces spliced together (splices are marked)
#   "annotated" -> one continuous span, first onset .. last offset
#   "full"      -> the entire wav, including long un-annotated stretches
COVER = "blocks"

# Block building (COVER="blocks"): calls closer than BLOCK_MERGE_S belong to the
# same block, and each block is padded by BLOCK_PAD_S on both sides.
BLOCK_MERGE_S = 0.5
BLOCK_PAD_S   = 0.25

# Fixed x-limits: every row shows WINDOW_S seconds, measured from the row start.
XLIM = (0.0, WINDOW_S)

# --- Sequence definition: consecutive calls separated by <= SEQ_GAP_S. ---
SEQ_GAP_S     = 0.035   # 35 ms
MIN_SEQ_CALLS = 2       # a "sequence" needs at least this many calls

# --- Recording location, read off the file name (..._channel_NN_...) ---
LOCATION_CHANNELS = {
    "underground": {30},
    "arena":       {10, 20},
}

# Every page is written here as well as shown inline.
EXPORT_DIR = REPO_ROOT / "exports" / "complex_calls_new_folders"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
SAVE_FMT = "png"
SAVE_DPI = 110

CALL_COLORS = {
    "warble":    "#e4572e",
    "high-freq": "#f3a712",
    "arc":       "#7d3ac1",
    "dwn":       "#1f6feb",
    "stack":     "#17a2b8",
}

print(f"every row spans {WINDOW_S:.1f} s; files wrap onto as many rows as they need "
      f"(cover={COVER!r})")
print(f"figures will be saved to: {EXPORT_DIR}")


## Loading

DAS writes one class-registration row per label (`NaN` start, `channel = -1`); those
are dropped — they are not real annotations.

In [ ]:
def load_annotations(csv_path: Path) -> pd.DataFrame:
    """Real annotations only, sorted by onset."""
    df = pd.read_csv(csv_path).dropna(subset=["start_seconds"])
    df = df.rename(
        columns={"name": "label", "start_seconds": "onset_s", "stop_seconds": "offset_s"}
    )
    return df.sort_values("onset_s").reset_index(drop=True)


def list_pairs(data_dir: Path) -> list[tuple[Path, pd.DataFrame]]:
    """(wav, annotations) pairs that actually have at least one annotation."""
    pairs, skipped = [], []
    for wav in sorted(data_dir.glob("*.wav")):
        csv = wav.with_name(wav.stem + "_annotations.csv")
        if not csv.exists():
            skipped.append((wav.name, "no annotation csv"))
            continue
        ann = load_annotations(csv)
        if ann.empty:
            skipped.append((wav.name, "no annotations"))
            continue
        pairs.append((wav, ann))
    if skipped:
        print(f"  skipped {len(skipped)} file(s): "
              + ", ".join(f"{n} ({why})" for n, why in skipped[:5])
              + (" ..." if len(skipped) > 5 else ""))
    return pairs


def channel_of(stem: str) -> int | None:
    """Recording channel parsed from the file name, e.g. exp_235_channel_30_file_043 -> 30."""
    m = re.search(r"channel_(\d+)", stem)
    return int(m.group(1)) if m else None


def split_by_location(pairs):
    """Split (wav, ann) pairs into {location: pairs} using LOCATION_CHANNELS."""
    out = {loc: [] for loc in LOCATION_CHANNELS}
    unknown = []
    for wav, ann in pairs:
        ch = channel_of(wav.stem)
        for loc, chans in LOCATION_CHANNELS.items():
            if ch in chans:
                out[loc].append((wav, ann))
                break
        else:
            unknown.append((wav.stem, ch))
    if unknown:
        print(f"  !! {len(unknown)} file(s) with an unrecognised channel: {unknown[:5]}")
    return out


seq_pairs  = list_pairs(SEQ_DIR)
sync_pairs = list_pairs(SYNC_DIR)
print(f"{SEQ_DIR.name}:  {len(seq_pairs)} files with annotations")
print(f"{SYNC_DIR.name}: {len(sync_pairs)} files with annotations")

seq_by_loc = split_by_location(seq_pairs)
print()
for loc, chans in LOCATION_CHANNELS.items():
    chan_str = ", ".join(f"channel_{c}" for c in sorted(chans))
    print(f"  {loc:12s} ({chan_str}): {len(seq_by_loc[loc])} files")


## Sequences: the 35 ms rule

A **sequence** is a run of consecutive calls where each gap (next onset − previous
offset) is `<= SEQ_GAP_S`. Below we check whether 35 ms is actually a sensible
cut-point in this data before relying on it.

In [ ]:
def call_gaps(ann: pd.DataFrame) -> np.ndarray:
    """Gaps between consecutive calls, in seconds. Negative = overlapping calls."""
    a = ann.sort_values("onset_s")
    return a["onset_s"].values[1:] - a["offset_s"].values[:-1]


def find_sequences(ann: pd.DataFrame, gap_s=SEQ_GAP_S, min_calls=MIN_SEQ_CALLS):
    """Runs of calls linked by gaps <= gap_s. Returns [(start_s, end_s, n_calls), ...]."""
    a = ann.sort_values("onset_s").reset_index(drop=True)
    if a.empty:
        return []
    cuts = np.where(call_gaps(a) > gap_s)[0]
    seqs = []
    for grp in np.split(np.arange(len(a)), cuts + 1):
        if len(grp) < min_calls:
            continue
        seqs.append(
            (float(a.loc[grp[0], "onset_s"]), float(a.loc[grp[-1], "offset_s"]), len(grp))
        )
    return seqs


In [ ]:
# --- Does 35 ms sit at a real trough in the gap distribution? ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, (pairs, name) in zip(axes, [(seq_pairs, "2026_07_10_new_data (sequences)"),
                                    (sync_pairs, "2026_07_15__synchronizing")]):
    g = np.concatenate([call_gaps(a) for _, a in pairs if len(a) >= 2])
    n_neg = int((g <= 0).sum())
    pos = g[g > 0] * 1000  # ms, log axis can't show <= 0

    ax.hist(pos, bins=np.logspace(np.log10(0.5), np.log10(6000), 60),
            color="0.35", edgecolor="none")
    ax.axvline(SEQ_GAP_S * 1000, color="#e4572e", lw=2,
               label=f"{SEQ_GAP_S * 1000:.0f} ms rule")
    ax.set_xscale("log")
    ax.set_xlabel("Gap between consecutive calls (ms, log)")
    ax.set_ylabel("count")
    frac = 100 * (g <= SEQ_GAP_S).sum() / len(g)
    ax.set_title(f"{name}\n{len(g)} gaps | {frac:.0f}% <= 35 ms"
                 + (f" | {n_neg} overlapping" if n_neg else ""), fontsize=10, loc="left")
    ax.legend()

fig.tight_layout()
fig.savefig(EXPORT_DIR / "gap_distribution_35ms_check.png", dpi=SAVE_DPI, bbox_inches="tight")
plt.show()


In [ ]:
# --- How much does the exact threshold matter? ---
rows = []
for thr_ms in [15, 25, 35, 50, 75]:
    n_seq = n_single = n_calls = 0
    lens = []
    for _, a in seq_pairs:
        n_calls += len(a)
        groups = find_sequences(a, gap_s=thr_ms / 1000, min_calls=1)
        for _, _, n in groups:
            lens.append(n)
            n_single += (n == 1)
        n_seq += len(groups)
    lens = np.array(lens)
    multi = lens[lens >= 2]
    rows.append({
        "gap_thr_ms": thr_ms,
        "n_groups": n_seq,
        "singletons": n_single,
        "n_sequences(>=2)": len(multi),
        "median_len": int(np.median(multi)),
        "max_len": int(lens.max()),
        "pct_calls_in_seq": round(100 * multi.sum() / n_calls, 1),
    })
pd.DataFrame(rows)


## Plotting

Every row spans exactly `WINDOW_S` seconds and each file is **wrapped across as many
rows as it needs**, so none of the audio is hidden. `set_xlim` is applied identically
to every row, so if a file ends mid-row that row just goes blank at the edge rather
than being rescaled — nothing is stretched to fit.

`COVER` controls how much of each file is drawn: `"annotated"` (first call to last
call) or `"full"` (the entire wav). `"full"` is honest but expensive here — one arena
file is 360 s of audio holding 6 calls, which alone is 90 near-empty rows.

In [ ]:
def build_blocks(ann, dur, *, merge_s=BLOCK_MERGE_S, pad_s=BLOCK_PAD_S):
    """Annotated stretches only. Calls within merge_s join one block; each is padded."""
    on = ann["onset_s"].values
    off = ann["offset_s"].values
    raw = []
    s, e = on[0], off[0]
    for i in range(1, len(ann)):
        if on[i] - e <= merge_s:
            e = max(e, off[i])
        else:
            raw.append((s, e))
            s, e = on[i], off[i]
    raw.append((s, e))
    return [(max(0.0, a - pad_s), min(dur, b + pad_s)) for a, b in raw]


def cover_blocks(ann, dur, cover=None):
    """The absolute time ranges to draw for one file, per the COVER mode."""
    cover = cover or COVER
    if cover == "full":
        return [(0.0, dur)]
    if cover == "annotated":
        return [(max(0.0, float(ann["onset_s"].min()) - PRE_S),
                 min(dur, float(ann["offset_s"].max()) + POST_S))]
    return build_blocks(ann, dur)


def splice(x, fs, blocks):
    """Concatenate the block slices. Returns (audio, bounds).

    bounds entries are (cat_start, cat_end, abs_start, abs_end) so we can map
    absolute annotation times into the spliced timeline and label the splices.
    """
    segs, bounds, c = [], [], 0.0
    for a, b in blocks:
        i0, i1 = max(0, int(a * fs)), min(x.shape[0], int(b * fs))
        if i1 <= i0:
            continue
        segs.append(x[i0:i1])
        span = (i1 - i0) / fs
        bounds.append((c, c + span, a, b))
        c += span
    audio = np.concatenate(segs) if segs else np.zeros(0, dtype=np.float32)
    return audio, bounds


def to_cat(t_abs, bounds):
    """Absolute time -> spliced-timeline time (None if it falls in cut-out silence)."""
    for cs, ce, a, b in bounds:
        if a - 1e-9 <= t_abs <= b + 1e-9:
            return cs + (t_abs - a)
    return None


In [ ]:
def plot_wrapped(pairs, title, slug, *, window_s=WINDOW_S, cover=None,
                 row_h=ROW_H, with_text=False, mark_sequences=True,
                 seq_gap_s=SEQ_GAP_S, save=True, show=True, fmt=None):
    """All of `pairs`, every file wrapped across as many rows as it needs.

    With COVER="blocks" only the annotated stretches are drawn; the silence
    between them is cut out and the remaining pieces spliced together. Splice
    points are marked with a dashed line labelled with the absolute time.
    """
    if not pairs:
        print(f"{slug}: nothing to plot")
        return None
    cover = cover or COVER

    # --- pass 1: work out the layout (and cache what pass 2 needs) ---
    prepared, total_rows = [], 0
    for wav, ann in pairs:
        fs, x = read_audio(wav)
        dur = x.shape[0] / fs
        blocks = cover_blocks(ann, dur, cover)
        audio, bounds = splice(x, fs, blocks)
        del x
        if not bounds:
            continue
        cat_dur = bounds[-1][1]
        n_rows = max(1, int(np.ceil(cat_dur / window_s)))
        prepared.append((wav, ann, fs, audio, bounds, dur, cat_dur, n_rows))
        total_rows += n_rows

    n = total_rows
    kept = sum(p[6] for p in prepared)
    print(f"{slug}: {len(pairs)} files -> {len(prepared)} drawn, {n} rows "
          f"({kept:.0f}s of audio, {row_h * n:.0f} in tall)")

    fig, axes = plt.subplots(n, 1, figsize=(13, row_h * n), squeeze=False,
                             gridspec_kw={"hspace": 0.75})
    axes = axes[:, 0]

    # --- pass 2: draw ---
    r_i = 0
    for wav, ann, fs, audio, bounds, dur, cat_dur, n_rows in prepared:
        seqs = find_sequences(ann, gap_s=seq_gap_s)

        # remap annotations + sequences onto the spliced timeline
        vis = ann.copy()
        vis["onset_s"]  = [to_cat(t, bounds) for t in ann["onset_s"]]
        vis["offset_s"] = [to_cat(t, bounds) for t in ann["offset_s"]]
        vis = vis.dropna(subset=["onset_s", "offset_s"])
        seqs_cat = []
        for a_s, a_e, cnt in seqs:
            cs, ce = to_cat(a_s, bounds), to_cat(a_e, bounds)
            if cs is not None and ce is not None:
                seqs_cat.append((cs, ce, cnt))

        for r in range(n_rows):
            ax = axes[r_i]; r_i += 1
            t0 = r * window_s

            i0 = int(t0 * fs)
            i1 = min(int((t0 + window_s) * fs), audio.shape[0])
            if i1 > i0:
                mesh = plot_spectrogram(
                    ax, audio[i0:i1], fs,
                    min_freq=FMIN_HZ, max_freq=FMAX_HZ,
                    vmin=VMIN_DB, vmax=VMAX_DB, t_start=0.0,
                )
                # vector PDFs of a pcolormesh are enormous -- rasterize the
                # spectrogram itself, leave text/annotations as vector
                mesh.set_rasterized(True)

            row = vis.copy()
            row["onset_s"]  = row["onset_s"]  - t0
            row["offset_s"] = row["offset_s"] - t0
            row = row[(row["offset_s"] > 0) & (row["onset_s"] < window_s)]
            plot_segments_overlay(ax, row, color_map=CALL_COLORS, with_text=with_text)

            if mark_sequences:
                trans = ax.get_xaxis_transform()
                for cs, ce, _c in seqs_cat:
                    a_r, b_r = cs - t0, ce - t0
                    if b_r < 0 or a_r > window_s:
                        continue
                    ax.add_patch(Rectangle(
                        (max(a_r, 0), 1.03), min(b_r, window_s) - max(a_r, 0), 0.10,
                        transform=trans, clip_on=False,
                        facecolor="#111111", edgecolor="none",
                    ))

            # splice markers: where a chunk of silence was cut out
            for cs, ce, abs_a, abs_b in bounds[1:]:
                xr = cs - t0
                if 0 < xr < window_s:
                    ax.axvline(xr, color="w", lw=1.4, alpha=0.9)
                    ax.axvline(xr, color="k", lw=0.8, ls=(0, (3, 2)))
                    ax.text(xr, 1.005, f" {abs_a:,.1f}s", transform=ax.get_xaxis_transform(),
                            fontsize=6, color="0.3", va="bottom", ha="left")

            ax.set_xlim(0.0, window_s)
            ax.set_xlabel("")
            ax.set_ylabel("kHz")
            # absolute time at the start of this row
            abs_t0 = next((a + (t0 - cs) for cs, ce, a, b in bounds if cs <= t0 < ce), None)
            ax.text(1.004, 0.5,
                    f"{abs_t0:,.1f}s" if abs_t0 is not None else "",
                    transform=ax.transAxes, fontsize=7, color="0.45",
                    va="center", ha="left")

            if r == 0:
                cut = dur - cat_dur
                t_first = float(ann["onset_s"].min())   # from the start of the wav
                ax.set_title(
                    f"{wav.stem}   |   annotations start @ {t_first:.2f}s   |   "
                    f"{len(ann)} calls, {dur:.1f}s file, "
                    f"{len(seqs)} seq (<={seq_gap_s * 1000:.0f}ms)   |   "
                    f"{len(bounds)} block(s), {cat_dur:.1f}s kept"
                    + (f", {cut:.1f}s silence cut" if cut > 0.05 else ""),
                    fontsize=9, loc="left", pad=24,
                )

    axes[-1].set_xlabel(f"Time within row (s)  —  each row spans {window_s:.0f} s")

    handles = [plt.Line2D([0], [0], color=c, lw=6, alpha=0.6, label=k)
               for k, c in CALL_COLORS.items()]
    if mark_sequences:
        handles.append(plt.Line2D([0], [0], color="#111111", lw=6,
                                  label=f"sequence (<={seq_gap_s * 1000:.0f}ms)"))
    if cover == "blocks":
        handles.append(plt.Line2D([0], [0], color="k", lw=1.2, ls=(0, (3, 2)),
                                  label="splice (silence cut)"))
    fig.legend(handles=handles, loc="upper right", ncol=len(handles), fontsize=9)
    fig.suptitle(f"{title}   ({len(prepared)} files, {n} rows, cover={cover!r})",
                 fontsize=12, x=0.01, ha="left")

    written = []
    if save:
        fmts = [fmt or SAVE_FMT] if isinstance(fmt or SAVE_FMT, str) else list(fmt)
        if "pdf" in fmts and row_h * n > 200:
            print(f"  !! figure is {row_h * n:.0f} in tall; PDF pages max out at 200 in")
        for f in fmts:
            out = EXPORT_DIR / f"{slug}.{f}"
            fig.savefig(out, dpi=SAVE_DPI, bbox_inches="tight")
            written.append(out)
            print(f"  saved {out.name} ({out.stat().st_size / 1e6:.1f} MB)")
    if show:
        plt.show()
    plt.close(fig)
    return written


---
## A. `2026_07_10_new_data` — putative call sequences, split by location

Split by recording channel, one output file each:

| file | channels | |
|:-|:-|:-|
| `new_data_underground.png` | `channel_30` | **underground** calls |
| `new_data_arena.png` | `channel_10`, `channel_20` | **arena** calls |

Every row still shares the same x-axis, so the two files are directly comparable to
each other as well as within themselves.

In [ ]:
# Underground — channel_30
underground_files = plot_wrapped(
    seq_by_loc["underground"],
    "2026_07_10_new_data — UNDERGROUND (channel_30)",
    slug="new_data_underground",
    fmt=("pdf", "png"),
)


In [ ]:
# Arena — channel_10 + channel_20
arena_files = plot_wrapped(
    seq_by_loc["arena"],
    "2026_07_10_new_data — ARENA (channel_10, channel_20)",
    slug="new_data_arena",
    fmt=("pdf", "png"),
)


---
## B. `2026_07_15__synchronizing_calls_examples` — two animals

The opposite case: calls that appear to come from **two different animals**. Labels are
drawn on this one (`with_text=True`) since the segments are what matter here.

In [ ]:
sync_file = plot_wrapped(
    sync_pairs,
    "2026_07_15__synchronizing  (two animals)",
    slug="synchronizing_two_animals",
    with_text=True,
)


---
## C. Overlapping calls

If two animals are calling, some annotations should **overlap in time** — one animal's
call starting before another's has finished. A single animal can't do that. This is a
quick check of how much overlap each folder actually contains.

In [ ]:
def overlap_summary(pairs, name):
    rows = []
    for wav, ann in pairs:
        a = ann.sort_values("onset_s").reset_index(drop=True)
        n_ov = 0
        for i in range(len(a) - 1):
            # does any later call start before this one ends?
            later = a.iloc[i + 1 :]
            n_ov += int((later["onset_s"] < a.loc[i, "offset_s"]).any())
        rows.append({"file": wav.stem, "n_calls": len(a), "n_overlapping": n_ov,
                     "frac": n_ov / max(1, len(a))})
    df = pd.DataFrame(rows)
    print(f"\n=== {name} ===")
    print(f"  files: {len(df)}   calls: {df['n_calls'].sum()}   "
          f"overlapping: {df['n_overlapping'].sum()} "
          f"({100 * df['n_overlapping'].sum() / max(1, df['n_calls'].sum()):.1f}% of calls)")
    return df.sort_values("frac", ascending=False)


seq_ov  = overlap_summary(seq_pairs,  "2026_07_10_new_data (sequences)")
sync_ov = overlap_summary(sync_pairs, "2026_07_15__synchronizing (two animals)")
sync_ov.head(15)
